# EYES-DEFY-ANEMIA — Segmentation Phase 2 sweep: BASE tier (6 of 18 combos)

One of **3 tier-specific notebooks** (Base / Mid / Strong), split out of the original single 18-combo notebook after a real Kaggle run failed with `Your notebook tried to use more disk space than is available` (20.93GB, 5h40m into a real run — full account in `Segmentation/.project_memory/kaggle/01_kaggle_notes.md`). Running each tier as its own session keeps peak disk usage to roughly a third of the full sweep, on top of two other real fixes already applied: `trainer_engine.py` now saves only ONE checkpoint per model (fp16, not fp32) instead of up to 3 fp32 checkpoints per model, and `sync_outputs()` no longer leaves a duplicate copy of everything behind after mirroring it.

**This notebook trains the 3 lightest architectures** (EfficientNet-B1 U-Net ~8.8M, SegFormer-B2 ~27.3M, CoAtNet-0 U-Net ~30.8M), each on both tissue types — 6 combos total, combo numbers 1-6 of the full 18. Run `segmentation-pretrained-sweep-mid.ipynb` and `segmentation-pretrained-sweep-strong.ipynb` (same folder) separately for the remaining 12. Full roster/rationale: `Segmentation/.project_memory/04_pretrained_architecture_sweep.md`.

## Setup

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

In [ ]:
# Diagnostic, not a hardcoded assumption -- Kaggle's actual dataset mount
# path does not always match its display name, and can be nested deeper
# than expected (this project has been bitten by this before, twice now:
# see classification/.project_memory/kaggle/01_kaggle_notes.md -- real
# paths have landed under /kaggle/input/datasets/<username>/<slug>/, not
# directly under /kaggle/input/<slug>/). Recurses a few levels deep so
# this is caught in one pass instead of needing to descend manually.
# Read the output below, THEN set the dataset dir variable(s) in the next cell.
import os


def print_tree(path, depth=0, max_depth=4):
    for entry in sorted(os.listdir(path)):
        full = os.path.join(path, entry)
        print("  " * depth + entry)
        if os.path.isdir(full) and depth < max_depth:
            print_tree(full, depth + 1, max_depth)


print_tree("/kaggle/input")

**Before running the next cell:** attach your uploaded dataset(s) to this notebook -- either both zips together as ONE Kaggle dataset, or as TWO SEPARATE datasets (this is what actually happened the first time this notebook was run: Kaggle listed them individually in the Input panel as `aligned_raw` and `aligned_raw_forniceal`). Either way, run the listing cell above, read its REAL printed output, and set the path(s) below from that -- **not** the placeholder text left in by default, and not a guessed path based on the dataset's display name (Kaggle's actual mount path does not always match it).

In [ ]:
# Confirmed real mount paths (project author's Kaggle account, verified via
# the print_tree() listing above on 2026-08-08 -- see
# Segmentation/.project_memory/kaggle/01_kaggle_notes.md for the full
# doubly-nested structure this came from). If you re-attach the datasets
# under a different username/slug, or Kaggle changes its mount scheme again,
# re-run the listing cell above and update these two lines from its real
# output -- don't guess.
ALIGNED_RAW_DATASET_DIR = "/kaggle/input/datasets/manivafapour21/aligned-raw"
ALIGNED_RAW_FORNICEAL_DATASET_DIR = "/kaggle/input/datasets/manivafapour21/aligned-raw-forniceal"

In [ ]:
# Only packages actually missing from Kaggle's base image (torch/torchvision,
# opencv, pandas, PIL, scikit-learn -- and therefore scipy, its own
# dependency -- are already there; scipy is listed explicitly anyway
# below rather than silently assumed, since it is now a real, load-bearing
# dependency for HD95/Wilcoxon in segmentation_metrics.py /
# compare_models_significance.py). Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations scipy segmentation-models-pytorch timm transformers einops

## Data

In [ ]:
import shutil
import zipfile
from pathlib import Path

DST_ROOT = Path("Segmentation/data/processed")


def stage_tissue_data(name: str, dataset_dir: str):
    """Copies {name}/ from the given Kaggle-attached dataset directory into
    Segmentation/data/processed/{name}/. Tries three possible layouts rather
    than assuming one, since Kaggle can present an uploaded zip differently
    depending on upload method:
      1. dataset_dir/{name}/images,masks/  -- the zip's own internal "{name}/"
         prefix preserved as-is (this is how aligned_raw.zip/
         aligned_raw_forniceal.zip were actually built -- see
         Segmentation/scripts/build_aligned_dataset{,_forniceal}.py).
      2. dataset_dir/images,masks/         -- Kaggle stripped/flattened that
         top-level folder on extraction.
      3. dataset_dir/{name}.zip            -- never auto-extracted at all,
         still sitting there as a raw zip file.
    """
    dataset_dir = Path(dataset_dir)
    dst = DST_ROOT / name
    shutil.rmtree(dst, ignore_errors=True)

    nested_dir = dataset_dir / name
    flat_zip = dataset_dir / f"{name}.zip"

    if nested_dir.is_dir():
        shutil.copytree(nested_dir, dst)
    elif (dataset_dir / "images").is_dir() and (dataset_dir / "masks").is_dir():
        shutil.copytree(dataset_dir, dst)
    elif flat_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(flat_zip) as zf:
            zf.extractall(DST_ROOT)  # zip's own internal paths already start with f"{name}/"
    else:
        raise FileNotFoundError(
            f"Could not find {name}/, images+masks/, or {name}.zip under {dataset_dir} -- "
            f"run the /kaggle/input listing cell above and check what's actually there."
        )

    n_images = len(list((dst / "images").glob("*.jpg")))
    n_masks = len(list((dst / "masks").glob("*.png")))
    print(f"{name}: {n_images} images, {n_masks} masks staged at {dst}")


stage_tissue_data("aligned_raw", ALIGNED_RAW_DATASET_DIR)
stage_tissue_data("aligned_raw_forniceal", ALIGNED_RAW_FORNICEAL_DATASET_DIR)

In [ ]:
# Combined sanity check, run BEFORE any real training:
#   1. Confirm the pip-installed heavy dependencies actually work -- a
#      dataloader-only check would NOT catch a missing/broken install here
#      (classification's own Kaggle notes: a plain dataloader check only
#      exercises dataset.py's imports, so a missing package silently
#      surfaces only much later, on the first real training script).
#   2. Confirm the 9-model registry itself imports cleanly.
#   3. Pull one real batch from BOTH tissue-type dataloaders.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("Segmentation/scripts").resolve()))
sys.path.insert(0, str(Path("Segmentation").resolve()))

import segmentation_models_pytorch as smp
import timm
import transformers
import einops

print("segmentation_models_pytorch", smp.__version__)
print("timm", timm.__version__)
print("transformers", transformers.__version__)
print("einops", einops.__version__)

from models.segmentation.pretrained_registry import ARCHITECTURE_REGISTRY
print(f"\n{len(ARCHITECTURE_REGISTRY)} architectures registered:")
for name in ARCHITECTURE_REGISTRY:
    print(" ", name)

from dataset import get_dataloaders

loaders = get_dataloaders()
for key in ["aligned_seg_train", "aligned_seg_forniceal_train"]:
    images, masks = next(iter(loaders[key]))
    print(f"\n{key}: image batch {tuple(images.shape)}, mask batch {tuple(masks.shape)}, "
          f"{len(loaders[key].dataset)} patients")

## Training

In [ ]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidates Segmentation/outputs/{checkpoints,logs,plots}/ into a single
    top-level /kaggle/working/outputs/ folder and re-zips it to
    /kaggle/working/segmentation_sweep_results.zip. Called after EVERY
    training cell below, not just at the end -- if the run gets cut short
    partway through the 18 combos, whatever completed so far is still
    cleanly consolidated and zipped, ready to download.

    IMPORTANT (added after a real Kaggle disk-quota crash, "Your notebook
    tried to use more disk space than is available", 20.93GB used, failed
    5h40m into a real run): each of the 18 training scripts can write up to
    3 full checkpoint files at fp32 (best-overall + one per loss function),
    and none of that is ever deleted between combos -- for the larger
    architectures (ConvNeXt-Large ~203M params, Swin-Large ~234M, etc.)
    those add up to hundreds of MB to ~1GB EACH. Originally this function
    copied Segmentation/outputs/ into /kaggle/working/outputs/ and left the
    source in place, so between the untouched source, the mirrored copy,
    and the zip made from that copy, roughly 3 copies of everything
    produced so far existed on disk simultaneously -- comfortably enough to
    blow a ~20GB quota partway through the Mid/Strong tiers. Now the
    source is deleted right after it's safely copied into
    /kaggle/working/outputs/, which becomes the single accumulating copy
    (plus the zip made from it) -- every training script recreates
    Segmentation/outputs/{checkpoints,logs,plots}/ fresh via its own
    mkdir(parents=True, exist_ok=True) on its next run, so nothing is lost
    by clearing the source here.
    """
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("Segmentation/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
            shutil.rmtree(src)
    archive_path = shutil.make_archive("/kaggle/working/segmentation_sweep_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

### Training — Base tier (6 combos, lightest)

In [ ]:
# Base tier, combo 1/18 -- EfficientNet-B1 U-Net (CNN, ~8.8M), palpebral
!python Segmentation/scripts/train_pretrained/train_cnn_base_efficientnet_b1_unet_palpebral.py
sync_outputs()

In [ ]:
# Base tier, combo 2/18 -- EfficientNet-B1 U-Net (CNN, ~8.8M), forniceal_palpebral
!python Segmentation/scripts/train_pretrained/train_cnn_base_efficientnet_b1_unet_forniceal_palpebral.py
sync_outputs()

In [ ]:
# Base tier, combo 3/18 -- SegFormer-B2 (Transformer, ~27.3M), palpebral
!python Segmentation/scripts/train_pretrained/train_transformer_base_segformer_b2_palpebral.py
sync_outputs()

In [ ]:
# Base tier, combo 4/18 -- SegFormer-B2 (Transformer, ~27.3M), forniceal_palpebral
!python Segmentation/scripts/train_pretrained/train_transformer_base_segformer_b2_forniceal_palpebral.py
sync_outputs()

In [ ]:
# Base tier, combo 5/18 -- CoAtNet-0 U-Net (Hybrid, ~30.8M), palpebral
!python Segmentation/scripts/train_pretrained/train_hybrid_base_coatnet0_unet_palpebral.py
sync_outputs()

In [ ]:
# Base tier, combo 6/18 -- CoAtNet-0 U-Net (Hybrid, ~30.8M), forniceal_palpebral
!python Segmentation/scripts/train_pretrained/train_hybrid_base_coatnet0_unet_forniceal_palpebral.py
sync_outputs()

## Done — what to download

Everything from this tier is consolidated at `/kaggle/working/outputs/` (checkpoints, logs) and zipped to `/kaggle/working/segmentation_sweep_results.zip`. Both are visible in this notebook version's **Output** tab once you Save Version -> Save & Run All — download the zip directly from there. Download and keep this zip somewhere safe (e.g. manually to Google Drive) before running the Mid-tier notebook in a fresh session, since each tier notebook's `/kaggle/working` starts empty.

A failed `!python ...` cell does **not** halt "Run All" — check each script's own printed output (or the saved `Segmentation/outputs/logs/*_study_summary.json` files) after this finishes, not just whether the notebook run itself completed.

In [ ]:
from pathlib import Path

print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/segmentation_sweep_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")